        # 02 FIRE 達成年齢の比較

        - 目的：**FIRE 目標資産**と**FIRE 達成年齢**を、貯蓄率・収益率・生活費を変えて比べる（RQ1〜RQ3）
        - 計算：`fire_analysis.py`、`optimization.py`

> 架空のデータによる**教育用シミュレーション**です。将来の投資収益の予測・保証や、投資・金融商品の推奨ではありません。金額は**万円**。税・社会保険料・インフレ・手数料・年金は考慮していません。

## 準備：実験環境（最初に必ず実行）

F2 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（F2 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F2 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（F2 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                               # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f2-financial-independence"   # リポジトリの中の F2 フォルダ（ルートが F2 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f2_root(p):
    return (Path(p) / "financial_model.py").exists() and (Path(p) / "fire_analysis.py").exists()


def find_f2_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F2 のルート
        if is_f2_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f2_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F2 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f2_upload")
    for hit in glob.glob("/content/f2_upload/**/financial_model.py", recursive=True):
        if is_f2_root(Path(hit).parent):
            return Path(hit).parent
    return None


F2_ROOT = find_f2_root()
if F2_ROOT is None:
    raise RuntimeError("F2 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F2_ROOT)
if str(F2_ROOT) not in sys.path:
    sys.path.insert(0, str(F2_ROOT))
if not (F2_ROOT / "textbook" / "max.py").exists():
    print("⚠️ textbook/ がありません。fire_analysis.py・optimization.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import financial_model as fm
import fire_analysis as fa
import retirement_model as rm
import optimization as op
import fire_charts as fc

fc.setup_style()                     # 文字を大きくするなら fc.setup_style(1.3)
scenarios, settings = fm.load_scenarios()
named = {s["name"]: s for s in scenarios}
END_ACC = settings["accumulation_end_age"]      # 資産形成の計算終了年齢（80）
END_RET = settings["retirement_end_age"]        # リタイア後の計算終了年齢（100）
W = settings["withdrawal_rate"]                 # 取り崩し率（4%、教育用の仮定）

SAVE_FIGURES = False                 # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", fc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    data = [{h: (("not reached" if k == "fire_age" else "-") if r.get(k) is None else f.format(r[k]))
             for k, h, f in columns} for r in rows]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(data))
    except ImportError:
        heads = [h for _k, h, _f in columns]
        widths = [max(len(h), *(len(d[h]) for d in data)) for h in heads]
        print("  ".join(h.rjust(w) for h, w in zip(heads, widths)))
        for d in data:
            print("  ".join(d[h].rjust(w) for h, w in zip(heads, widths)))


print("F2_ROOT:", F2_ROOT)
print("シナリオ:", [s["label"] for s in scenarios], f"/ 取り崩し率 {W:.0%} / 資産形成 {END_ACC} 歳まで / リタイア後 {END_RET} 歳まで")

## FIRE 目標資産と達成年齢

$$ \text{FIRE Target} = \frac{\text{Annual Expenses}}{\text{Withdrawal Rate}} \qquad \text{例：} \frac{300}{0.04} = 7{,}500 \text{ 万円} $$

- **FIRE 達成年齢**：年齢別資産を若い順に調べ、初めて目標以上になった年齢（**線形探索**、教科書 `seq_search`）
- 80 歳までに到達しなければ **未達成（None）**
- 取り崩し率 4% は**教育用の仮定**で、安全を保証する値ではありません（README 7章）

In [ ]:
print("FIRE 目標（生活費 300 万円、4%）=", fa.fire_target(300, 0.04), "万円")
results = [op.evaluate_scenario(s, settings) for s in scenarios]
table(results, [("label", "scenario", "{}"), ("savings_rate", "savings", "{:.0%}"), ("annual_return", "return", "{:.0%}"),
                ("annual_expenses", "expenses", "{:,.0f}"), ("target", "FIRE target", "{:,.0f}"),
                ("fire_age", "FIRE age", "{}"), ("assets_at_report", "assets at 65", "{:,.0f}")])

## 年齢別資産と FIRE 目標ライン

シナリオごとに目標が違うので、各シナリオの「目標に対する割合」を描きます（1.0 の線が目標）。点が FIRE 達成年齢です。

In [ ]:
#@title グラフを表示（コードは隠しています）
series, fire_ages = [], []
for s in scenarios:
    rws = fm.accumulate(s, END_ACC)
    a, v = fm.assets_by_age(rws)
    t = fa.fire_target(s["annual_expenses"], W)
    series.append((s["label"], a, [x / t for x in v], fc.color_of(s["name"])))
    fire_ages.append(fa.fire_age(rws, t))
fig = fc.plot_assets(series, "Assets relative to each FIRE target (1.0 = target)", target=1.0,
                     fire_ages=fire_ages, ylabel="assets / FIRE target", target_label="FIRE target (= 1.0)")
show(fig, "02_assets_vs_target")

## 1 つのシナリオを詳しく：資産と目標ライン

In [ ]:
s = named["standard"]                            # TODO：他のシナリオに変えてみよう
rws = fm.accumulate(s, END_ACC)
t = fa.fire_target(s["annual_expenses"], W)
a, v = fm.assets_by_age(rws)
age = fa.fire_age(rws, t)
show(fc.plot_assets([(s["label"], a, v, fc.color_of(s["name"]))], f"{s['label']}: assets and FIRE target",
                    target=t, fire_ages=[age]), "02_standard_target")
print("FIRE 達成年齢（線形探索）:", age, " / 二分探索:", fa.fire_age_binary(rws, t))

## 貯蓄率・収益率・生活費と FIRE 達成年齢

灰色の帯の × は **80 歳までに未達成**（年齢の値ではありません）。

In [ ]:
#@title グラフを表示（コードは隠しています）
base = named["standard"]
rates = op.savings_rate_candidates()
returns = op.return_candidates()
expenses = [200, 240, 280, 320, 360]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
r1 = op.sweep_savings_rate(base, rates, settings, keep="income")
fc.plot_fire_age(rates, [r["fire_age"] for r in r1], "savings rate (income fixed)", "Savings rate", END_ACC, ax=axes[0])
r2 = op.sweep_return(base, returns, settings)
fc.plot_fire_age(returns, [r["fire_age"] for r in r2], "annual return", "Annual return", END_ACC, ax=axes[1])
r3 = op.sweep_expenses(base, expenses, settings, keep="income")
fc.plot_fire_age(expenses, [r["fire_age"] for r in r3], "annual expenses (income fixed)", "Annual expenses",
                 END_ACC, xfmt="{}", ax=axes[2])
fig.tight_layout()
show(fig, "02_fire_age_vs_params")

## 生活費の実験は 2 種類ある（RQ3）

| 実験 | 固定するもの | 変わるもの |
| --- | --- | --- |
| 所得を固定 | 年間所得 400 万円 | 生活費を変えると、貯蓄率も変わる |
| 貯蓄率を固定 | 貯蓄率 20% | 生活費を変えると、所得も変わる |

In [ ]:
for keep in ["income", "savings_rate"]:
    rows = op.sweep_expenses(base, expenses, settings, keep=keep)
    table(rows, [("param", "expenses", "{}"), ("annual_income", "income", "{:,.0f}"),
                 ("savings_rate", "savings rate", "{:.0%}"), ("target", "FIRE target", "{:,.0f}"),
                 ("fire_age", "FIRE age", "{}")])
# TODO：2 つの表の FIRE 達成年齢を比べ、違い（または同じになる理由）を式から考えよう

## 「最適」な貯蓄率は？

候補の中で FIRE 達成年齢が最小になる条件を選びます（教科書 `max_of` で、符号を反転した最大値 → `seq_search` で位置）。
**ただし、貯蓄率を上げるほど今の生活費は減ります。** 何を最適とするかはチームで決めてください。

In [ ]:
b = op.best(r1, "fire_age", minimize=True)
print(f"FIRE 達成年齢が最小：貯蓄率 {b['param']:.0%}（{b['fire_age']} 歳）、現役時の生活費 {b['current_expenses']:,.0f} 万円")
table(r1, [("param", "savings rate", "{:.0%}"), ("current_expenses", "current expenses", "{:,.0f}"),
           ("fire_age", "FIRE age", "{}")])
# TODO：最低生活費（例：年 250 万円）の制約を置くと、選ばれる貯蓄率はどう変わるか（Advanced C）

## 学生向け TODO・考察

- [ ] 生活費・貯蓄率・収益率・取り崩し率（`W`）を変えて FIRE 達成年齢を比較する
- [ ] 貯蓄率と収益率のどちらが FIRE 達成年齢に大きく影響するか、仮説を立てて調べる
- [ ] `fa.fire_age` と `fa.fire_age_binary` の違い（二分探索が使える条件）を説明する

**考察（チームで記入）**：TODO

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索・二分探索、第6章 挿入ソート）
- F2 リポジトリの `README.md`（モデルの仮定、4% ルールとその限界、計算量、モデルの限界と倫理的配慮）
- データ：`data/sample_scenarios.json`（この教材のための**架空のデータ**）